[Guide Index](README.md)


# 02 Graphs

`StarLayerGraph` is a wrapper over a `pyoxigraph.Store` that supports RDF 1.2: triple terms, reification, and direction-tagged literals. Oxigraph stores these natively; the wrapper adds the StarLayer helpers around them (`add_reification`, `reifiers`, `triple_terms`, prefix handling, pretty Turtle, set operators). `StarLayerGraph` is the working name of the class that becomes `StarLayerGraph` when the Oxigraph baseline replaces the earlier rdflib-based one.

pyoxigraph's classes cannot be subclassed, so `StarLayerGraph` holds the store rather than extending it, and it is not an `rdflib.Graph`. Terms that come out of an `StarLayerGraph` are pyoxigraph terms (`NamedNode`, `BlankNode`, `Literal`, `Triple`); `.n3()` does not exist on them, so use `g.qname_term(x)` for prefixed output or `str(x)` for the N-Triples form. rdflib terms are still accepted as input.

A related guide goes into more depth regarding datasets: **[2.a Working with datasets](02a-graphs-datasets.ipynb)** (`StarLayerDataset`, the working name of `StarLayerDataset`). The boundary between the wrapper and Oxigraph is covered in **[2.d The Oxigraph-backed graph](02d-oxigraph-wrapper.ipynb)**.

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). 

Run cells from top to bottom; later sections may reuse variables from earlier ones.


In [1]:
import pyoxigraph as ox
from pyoxigraph import Literal, Triple

from starlayer.graph.oxgraph import StarLayerGraph


class Ns:
    """A tiny namespace helper: Ns(base).name is a pyoxigraph NamedNode."""
    def __init__(self, base): self.base = base
    def __getattr__(self, name): return ox.NamedNode(self.base + name)


EX = Ns("http://example.org/")
RDF = Ns("http://www.w3.org/1999/02/22-rdf-syntax-ns#")

## 1. RDF 1.2 semantics

Oxigraph, and so `StarLayerGraph`, supports RDF 1.2:
- Triple terms (`pyoxigraph.Triple`) and statement resources
- Reification via `rdf:reifies` and statement metadata
- Direction-tagged strings such as `"hello"@en--ltr` and `"مرحبا"@ar--rtl`
- Parsing and serialization of RDF 1.2 using Turtle 1.2 syntax.

Prefixes are not part of the store. A new graph has none bound; `g.bind(prefix, namespace)` records one in the graph's prefix dictionary (`g.prefixes`, also `g.namespaces()`), and the dictionary is passed to Oxigraph on each `serialize()` and `query()` call.

In [2]:
# create the graph, and bind a prefix (a new graph has none)
g = StarLayerGraph()
g.bind("ex", EX.base)

# create a triple term (a pyoxigraph Triple)
tt = Triple(EX.alice, EX.worksFor, EX.AcmeCorp)

# create a reifier (ex:claim) associated with the triple term and add it to the graph
g.add_reification(EX.claim, tt)
g.add((EX.claim, EX.source, EX.HRSystem))

# serialize(format="turtle") is Oxigraph's flat Turtle 1.2 writer; pretty=True is StarLayer's grouping writer,
# which folds the reifier into <<  >> syntax.
print("\nResulting graph")
print(g.serialize(format="turtle", pretty=True))

# note that creating a reifier for a triple term, does not automatically assert that term into the graph.
# ex:alice ex:worksFor ex:AcmeCorp is not asserted.
print("asserted:", (EX.alice, EX.worksFor, EX.AcmeCorp) in g)


Resulting graph
@version "1.2" .
@prefix ex: <http://example.org/> .

<< ex:alice ex:worksFor ex:AcmeCorp ~ ex:claim >> ex:source ex:HRSystem .

asserted: False


In [3]:
g = StarLayerGraph()
g.bind("ex", EX.base)
g.bind("rdf", RDF.base)

# add reifiers to the graph; a 3-tuple as the object is converted to a triple term
g.add((EX.claim, RDF.reifies, (EX.alice, EX.worksFor, EX.AcmeCorp)))
g.add((EX.other, RDF.reifies, (EX.alice, EX.likes, EX.ProductABC)))

# explicitly adding a triple to the graph.
g.add((EX.alice, EX.worksFor, EX.GlobalTech))

# triples() accepts a triple term as the object when selecting triples
# selects all triples with the triple term as the object
selectTriples = g.triples((None, None, (EX.alice, EX.worksFor, EX.AcmeCorp)))

for s, p, o in selectTriples:
    print(g.qname_term(s), g.qname_term(p), g.qname_term(o))
for t in g.triple_terms(subject=EX.alice):
    print(g.qname_term(t))

# has_triple_term() tests whether the triple term is in the graph.
# (EX.alice, EX.worksFor, EX.GlobalTech) is asserted directly in the graph, but is not the
# object of any triple, so it returns False.
print(g.has_triple_term(EX.alice, EX.worksFor, EX.AcmeCorp))
print(g.has_triple_term(EX.alice, EX.worksFor, EX.GlobalTech))

print("\nResulting graph")
print(g.serialize(format="turtle", pretty=True))

ex:claim rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>>
<<( ex:alice ex:likes ex:ProductABC )>>
<<( ex:alice ex:worksFor ex:AcmeCorp )>>
True
False

Resulting graph
@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:alice ex:worksFor ex:GlobalTech .

ex:claim rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> .

ex:other rdf:reifies <<( ex:alice ex:likes ex:ProductABC )>> .



In [4]:
# continues using g from the previous cell

# rdf:reifies is the common approach to making a statement about a triple statement.
# RDF 1.2 allows triple terms in the object position of any triple. (rdf:reifies is not a required predicate)

# add a triple to the graph with a triple term as the object
g.add((EX.AuditSystem, EX.reported, (EX.alice, EX.worksFor, EX.AcmeCorp)))

# triples() accepts a triple term as object to select matching triples
selectTriples = g.triples((None, None, (EX.alice, EX.worksFor, EX.AcmeCorp)))

# qname_term() is a starlayer function that adds qname transformation to terms, triple terms included.
for s, p, o in selectTriples:
    print(g.qname_term(s), g.qname_term(p), g.qname_term(o))

print("\nResulting graph")
print(g.serialize(format="turtle", pretty=True))

ex:AuditSystem ex:reported <<( ex:alice ex:worksFor ex:AcmeCorp )>>
ex:claim rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>>

Resulting graph
@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:AuditSystem ex:reported <<( ex:alice ex:worksFor ex:AcmeCorp )>> .

ex:alice ex:worksFor ex:GlobalTech .

ex:claim rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> .

ex:other rdf:reifies <<( ex:alice ex:likes ex:ProductABC )>> .



### 1.1 Working with statements about statements

`StarLayerGraph` adds methods for handling reifications and navigating the reifier/triple-term/annotation relationships (the same functions are available as `starlayer.graph.oxgraph.reification.*`):
- `reifiers()`,
- `reifications()`,
- `reifier_annotations()`,
- `reified_triples()`, and
- `remove_reification()`

`remove_reification(reifier, triple_term=None)` can be scoped to one specific reifier (triple term), leaving any other triple(s) the same reifier reifies untouched; omit `triple_term` for the all reifier behavior.

In [5]:
# continues using g from the previous cell

# adds assertions to the reification ex:claim
g.add((EX.claim, EX.source, EX.HRSystem))

tt1 = (EX.alice, EX.worksFor, EX.AcmeCorp)
tt2 = (EX.alice, EX.likes, EX.ProductABC)

# reifiers(): returns the reifier node(s) that reify a given triple term.
print([g.qname_term(r) for r in g.reifiers(TT=tt1)])

# reifications(): returns the triple terms that have at least one reifier.
for tt in g.reifications():
    print(g.qname_term(tt))

# reifier_annotations(): returns a reifier's annotation triples (excludes rdf:reifies itself)
for reifier, pred, val in g.reifier_annotations(tt1):
    print(g.qname_term(reifier), g.qname_term(pred), g.qname_term(val))

# reified_triples(): returns the triple term(s) a specific reifier reifies
for tt in g.reified_triples(EX.claim):
    print(g.qname_term(tt))

# give ex:claim a second rdf:reifies link, so scoped vs. wildcard removal are distinguishable
g.add((EX.claim, RDF.reifies, tt2))

# remove_reification(reifier, triple_term): remove the reification link between a reifier
# and the specified triple term only.
g.remove_reification(EX.claim, tt1)
print((EX.claim, RDF.reifies, tt1) in g)         # False - only this link removed
print((EX.claim, RDF.reifies, tt2) in g)         # True  - untouched
print((EX.claim, EX.source, EX.HRSystem) in g)  # True  - untouched

# remove_reification(reifier): removes all rdf:reifies links from the reifier
g.remove_reification(EX.claim)
print((EX.claim, RDF.reifies, tt2) in g)
print((EX.claim, EX.source, EX.HRSystem) in g)

print("\nResulting graph")
print(g.serialize(format="turtle", pretty=True))

['ex:claim']
<<( ex:alice ex:likes ex:ProductABC )>>
<<( ex:alice ex:worksFor ex:AcmeCorp )>>
ex:claim ex:source ex:HRSystem
<<( ex:alice ex:worksFor ex:AcmeCorp )>>
False
True
True
False
True

Resulting graph
@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:AuditSystem ex:reported <<( ex:alice ex:worksFor ex:AcmeCorp )>> .

ex:alice ex:worksFor ex:GlobalTech .

ex:claim ex:source ex:HRSystem .

ex:other rdf:reifies <<( ex:alice ex:likes ex:ProductABC )>> .



### 1.2 Direction-tagged string literals

Oxigraph supports the directional language strings introduced in RDF 1.2 natively: `pyoxigraph.Literal(value, language=..., direction=...)`. (The earlier rdflib-based classes needed a separate `DirLangString` class; the wrapper does not.)

In [6]:
g = StarLayerGraph()
g.bind("ex", EX.base)

# literals can include language direction
g.add((EX.ProductABC, EX.description, Literal("مرحبا", language="ar", direction=ox.BaseDirection.RTL)))
g.add((EX.ProductABC, EX.description, Literal("hello", language="en")))
g.add((EX.ProductABC, EX.description, Literal("hello", language="en", direction=ox.BaseDirection.LTR)))

print(g.serialize(format="turtle", pretty=True))

@version "1.2" .
@prefix ex: <http://example.org/> .

ex:ProductABC ex:description "hello"@en--ltr, "hello"@en, "مرحبا"@ar--rtl .



## 2. Turtle 1.2 syntax

Oxigraph parses Turtle 1.2, including triple terms and directed language strings, and `format="turtle"` means Turtle 1.2 on this class (`turtle12` and `ttl` are accepted as aliases). A parse is atomic: a syntax error leaves the graph unchanged.

This section shows uses of Turtle 1.2 syntax. Output is made with `serialize(format="turtle", pretty=True)`, which groups and folds reifiers into `<< >>` and `{| |}` syntax to make it easier to understand the output graph. The earlier `longturtle12` format name is not offered by the wrapper; its one-triple-per-line layout is what Oxigraph's own writer (`serialize(format="turtle")`) produces.

See the [serialization formats guide](06a-serialization-formats.ipynb) for other RDF 1.2 serialization formats.

### 2.1 Canonical form: `rdf:reifies <<( s p o )>>`

The canonical form — a triple term in `<<( )>>` syntax, reified by name via `rdf:reifies.`

In [7]:
g = StarLayerGraph()
g.bind("ex", EX.base)
g.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
    ex:claim rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> ;
      ex:source ex:HRSystem .
""", format="turtle")

print("triples:", len(g))
print("\nResulting graph")
print(g.serialize(format="turtle", pretty=True))

# note that the triple "ex:alice ex:worksFor ex:AcmeCorp" is never asserted into the graph.


triples: 2

Resulting graph
@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

<< ex:alice ex:worksFor ex:AcmeCorp ~ ex:claim >> ex:source ex:HRSystem .



### 2.2 Reifying shorthand using: `<< s p o >>`

`<<( s p o )>>` above represents the triple *term* itself.  

`<< s p o >>` (no parens) is different: it is shorthand for creating an anonymous reifier with the provided predicate and object.  Similar to above, it does not assert the triple being reified. 

In [8]:
g = StarLayerGraph()
g.bind("ex", EX.base)
g.parse(data="""
    @prefix ex: <http://example.org/> .
    << ex:alice ex:worksFor ex:AcmeCorp >> ex:source ex:HRSystem .
""", format="turtle")

print("triples:", len(g))
print("\nResulting graph")
print(g.serialize(format="turtle", pretty=True))


triples: 2

Resulting graph
@version "1.2" .
@prefix ex: <http://example.org/> .

<< ex:alice ex:worksFor ex:AcmeCorp >> ex:source ex:HRSystem .



### 2.3 Naming the reifier: `<< s p o ~ ex:id >>`

Adding an IRI preceded by `~` to the shorthand, such as `<< s p o ~ ex:id >>`, allows the addition of a named reifier to the triple term. This allows for the adding of additional triples to the reifier, which is not possible with an anonymous reifier.

In [9]:
g = StarLayerGraph()
g.bind("ex", EX.base)
g.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
    << ex:alice ex:worksFor ex:AcmeCorp ~ ex:id >> ex:source ex:HRSystem;  ex:reportedDate "2026-01-01"^^xsd:date .
   
""", format="turtle")

print("triples:", len(g))
print("\nResulting graph")
print(g.serialize(format="turtle", pretty=True))

triples: 3

Resulting graph
@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<< ex:alice ex:worksFor ex:AcmeCorp ~ ex:id >> ex:reportedDate "2026-01-01"^^xsd:date ;
    ex:source ex:HRSystem .



### 2.4 Nesting a reifiedTriple

A `reifiedTriple` can be nested inside another — the object of one `<< >>` can itself be another `<< >>`, reifying a claim about a claim.

In [10]:
g = StarLayerGraph()
g.bind("ex", EX.base)
g.parse(data="""
    @prefix ex: <http://example.org/> .
    << ex:claim ex:about << ex:alice ex:worksFor ex:AcmeCorp >> ~ ex:outer >> ex:recordedBy ex:AuditSystem .
""", format="turtle")

print("triples:", len(g))
print("\nResulting graph")
print(g.serialize(format="turtle", pretty=True))

triples: 3

Resulting graph
@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

_:b0 rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> .

<< ex:claim ex:about _:b0 ~ ex:outer >> ex:recordedBy ex:AuditSystem .



### 2.5 Grouping references to triple term reifiers

A comma-separated list of `<< s p o >>` values lets one subject/predicate point at several *distinct* triple terms at once, creating a way to group triple terms together through their reifiers.  

In [11]:
g = StarLayerGraph()
g.bind("ex", EX.base)
g.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:report123 ex:contains << ex:sensor1 ex:measured "20.1" ~ ex:r1 >> ,
                              << ex:sensor2 ex:measured "22.3" ~ ex:r2 >> .
    ex:r1 ex:source ex:SystemA .
    ex:r2 ex:source ex:SystemB .
""", format="turtle")



print("\nThe triples ex:report123 contains:")
for s, p, reifier in g.triples((EX.report123, None, None)):
    for tt in g.reified_triples(reifier):
        print(" ", g.qname_term(tt))

# assert the triples in the report
for s, p, reifier in list(g.triples((EX.report123, None, None))):
    for tt in list(g.reified_triples(reifier)):
        g.add((tt.subject, tt.predicate, tt.object))

print("triples:", len(g))
print("\nResulting graph")
print(g.serialize(format="turtle", pretty=True))


The triples ex:report123 contains:
  <<( ex:sensor2 ex:measured "22.3" )>>
  <<( ex:sensor1 ex:measured "20.1" )>>
triples: 8

Resulting graph
@version "1.2" .
@prefix ex: <http://example.org/> .

ex:report123 ex:contains ex:r1, ex:r2 .

ex:sensor1 ex:measured "20.1" ~ ex:r1 {| ex:source ex:SystemA |} .

ex:sensor2 ex:measured "22.3" ~ ex:r2 {| ex:source ex:SystemB |} .



### 2.6 Inline annotation shorthand: `{| pred val |}`

The `{| ... |}` shorthand creates annotation triples at the same time that a triple is created.   

This example creates an anonymous reifier and attaches multiple annotations to that reifier.  

In [12]:
g = StarLayerGraph()
g.bind("ex", EX.base)
g.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice ex:likes ex:ProductABC {| ex:since "2020" ; ex:source ex:CRM |} .
""", format="turtle")

print("triples:", len(g))
print("\nResulting graph")
print(g.serialize(format="turtle", pretty=True))


# note: ex:alice ex:likes ex:ProductABC {| ex:since "2020" ; ex:source ex:CRM |} asserts and reifies
# the triple at the same time.  ex:alice ex:likes ex:ProductABC appears in the graph.

triples: 4

Resulting graph
@version "1.2" .
@prefix ex: <http://example.org/> .

ex:alice ex:likes ex:ProductABC {| ex:since "2020" ; ex:source ex:CRM |} .



### 2.7 Naming the reifier in an annotation block

The same example as above using `~` to create a named reifier. 


In [13]:
g = StarLayerGraph()
g.bind("ex", EX.base)
g.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice ex:likes ex:ProductABC ~ ex:id {| ex:since "2020" ; ex:source ex:CRM |} .
""", format="turtle")

print("triples:", len(g))
print("\nResulting graph")
print(g.serialize(format="turtle", pretty=True))


# ~ ex:id is created as a named entity as the reifier.

triples: 4

Resulting graph
@version "1.2" .
@prefix ex: <http://example.org/> .

ex:alice ex:likes ex:ProductABC ~ ex:id {| ex:since "2020" ; ex:source ex:CRM |} .



### 2.8 Multiple annotation blocks

The same example as above expanded to use multiple annotation blocks.


In [14]:
g = StarLayerGraph()
g.bind("ex", EX.base)
g.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice ex:likes ex:ProductABC  ~ ex:id1 {| ex:since "2020" ; ex:source ex:CRM |} ~ ex:id2 {| ex:since "2019" ; ex:source ex:alice |}.
""", format="turtle")

print("triples:", len(g))
print("\nResulting graph")
print(g.serialize(format="turtle", pretty=True))


# Each block/reifier here produces its own independent reification of the *same*
# underlying triple (ex:alice ex:likes ex:ProductABC) - two separate claims about one fact,


triples: 7

Resulting graph


@version "1.2" .
@prefix ex: <http://example.org/> .

ex:alice ex:likes ex:ProductABC ~ ex:id1 {| ex:since "2020" ; ex:source ex:CRM |} ~ ex:id2 {| ex:since "2019" ; ex:source ex:alice |} .



### 2.9 Direction-tagged literals: `"text"@lang--dir`

The same directional values built with `pyoxigraph.Literal` above are written directly in Turtle 1.2 as an ordinary language-tagged literal with a `--ltr`/`--rtl` direction suffix. Language tags are lower-cased by the store (`@EN-us` is stored as `@en-us`).

In [15]:
g = StarLayerGraph()
g.bind("ex", EX.base)
g.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:listing_en ex:description "hello"@en--ltr .
    ex:listing_ar ex:description "مرحبا"@ar--rtl .
""", format="turtle")

print("triples:", len(g))
print(g.serialize(format="turtle", pretty=True))
for o in g.objects(EX.listing_ar, EX.description):
    print(repr(o.value), o.language, o.direction)

triples: 2
@version "1.2" .
@prefix ex: <http://example.org/> .

ex:listing_ar ex:description "مرحبا"@ar--rtl .

ex:listing_en ex:description "hello"@en--ltr .

'مرحبا' ar rtl
